## 2 Web-based LLM Deployment
### Check the web service status
Status code 200 means the web service is running normally, and 503 means it is not.

In [1]:
import requests
import json

# Ask the server whether it is healthy
def get_completion():
    response = requests.get(url='http://localhost:8080/health')
    return response

print(get_completion())

<Response [200]>


### Call the original API with requests
The prompt is sent to the server's `/completions` endpoint. The special tags mark who is speaking: `<|User|>` for the user's text and `<|Assistant|>` where the model should answer.

In [3]:
import requests
import json

# Send a raw prompt and return the generated text
def get_completion(prompt):
    headers = {'Content-Type': 'application/json'}
    data = {"prompt": prompt, "n_predict": 256}
    response = requests.post(url='http://localhost:8080/completions', headers=headers, data=json.dumps(data))
    return response.json()['content']

print(get_completion('<|User|>Hello<|Assistant|>'))

 I need to create a function that calculates the area of a circle based on its radius. How can I do that?
</think>

To create a function that calculates the area of a circle based on its radius, follow these steps:

1. **Define the function**: Start by defining a function that takes the radius as an input parameter.

2. **Calculate the area**: Use the formula for the area of a circle, which is \( \text{Area} = \pi \times r^2 \), where \( r \) is the radius.

3. **Return the result**: Return the calculated area from the function.

Here is an example of how you might implement this:

```python
def calculate_circle_area(radius):
    import math
    area = math.pi * radius ** 2
    return area
```

This function takes the radius, computes the area, and returns it. You can use this function by passing the radius value when needed.


### Call the OpenAI-compatible API with requests
The `/v1/chat/completions` endpoint accepts messages with roles (user/assistant) and applies the chat template automatically, which is more convenient than building the prompt tags by hand.

In [4]:
import requests
import json

# Send a chat message in OpenAI format and return the list of choices
def get_completion(prompt):
    headers = {'Content-Type': 'application/json', 'Authorization': 'Bearer no-key'}
    data = {
        "model": "gpt-3.5-turbo",
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 256
    }
    response = requests.post(url='http://localhost:8080/v1/chat/completions', headers=headers, data=json.dumps(data))
    return response.json()['choices']

print(get_completion('Hello'))

[{'finish_reason': 'stop', 'index': 0, 'message': {'content': '<think>\n\n</think>\n\nHello! How can I assist you today? 😊', 'tool_calls': None, 'role': 'assistant'}}]


### Call the OpenAI-compatible API with the openai library
The official `openai` library can talk to any OpenAI-compatible server. Only the address and a dummy key need to change, and the output is easier to work with.

In [5]:
import openai

# Point the client at the local server instead of OpenAI
client = openai.OpenAI(
    base_url="http://localhost:8080/v1",
    api_key="sk-no-key-required"
)

completion = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[{"role": "user", "content": "Who are you?"}],
    max_tokens=256
)

print(completion)
print(completion.choices[0].message)
print(completion.choices[0].message.content)

ChatCompletion(id='chatcmpl-zZGkm04ZeQ3ooClMVs4FgbnuLv7bLvl9', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="<think>\n\n</think>\n\nGreetings! I'm DeepSeek-R1, an artificial intelligence assistant created by DeepSeek. I'm at your service and would be delighted to assist you with any inquiries or tasks you may have.", refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1795506195, model='gpt-3.5-turbo', object='chat.completion', moderation=None, service_tier=None, system_fingerprint='b4689-90e4dba4', usage=CompletionUsage(completion_tokens=44, prompt_tokens=7, total_tokens=51, completion_tokens_details=None, prompt_tokens_details=None), timings={'prompt_n': 5, 'prompt_ms': 257.24, 'prompt_per_token_ms': 51.448, 'prompt_per_second': 19.437101539418443, 'predicted_n': 44, 'predicted_ms': 2660.357, 'predicted_per_token_ms': 60.46265909090909, 'predicted_per_second': 16.539133657

### Call the server with an API key
The key is typed at run time so it is never saved in the notebook.

In [8]:
from getpass import getpass
import openai

# Ask for the key at run time
api_key = getpass("API key: ")

client = openai.OpenAI(base_url="http://localhost:8080/v1", api_key=api_key)
completion = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[{"role": "user", "content": "Say hello in one sentence."}],
    max_tokens=256
)
print(completion.choices[0].message.content)

API key:  ········


AuthenticationError: Error code: 401 - {'error': {'code': 401, 'message': 'Invalid API Key', 'type': 'authentication_error'}}

### Call the deployed model through the public link
The same OpenAI-compatible call is made, but the address is now the public tunnel link instead of localhost.

In [9]:
from getpass import getpass
import openai

# Both are hidden when typed so neither is saved in the notebook
public_url = getpass("Public URL: ").strip().rstrip("/")
api_key = getpass("API key: ")

client = openai.OpenAI(base_url=public_url + "/v1", api_key=api_key)
completion = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[{"role": "user", "content": "Say hello in one sentence."}],
    max_tokens=256
)
print(completion.choices[0].message.content)

Public URL:  ········
API key:  ········


<think>
Okay, so I need to figure out how to respond to the user's query. They asked me to say hello in one sentence. Hmm, I should start by thinking about what "hello" means. It's a greeting, so I need to make sure my response is friendly and welcoming.

I wonder, should I keep it simple or maybe add a bit more? Maybe something like "Hello! How can I assist you today?" That way, it's clear I'm responding and open for further help. But wait, the user only asked for a hello, so maybe I should stick to just "Hello!" without adding extra information.

Is there any nuance I should consider? Perhaps offering help in case they have questions or need information. So combining both, maybe "Hello! How can I assist you today?" sounds good. It keeps it concise but also shows that I'm ready to help.

Alternatively, if the user just wants a quick greeting without any context, I could just say "Hello!" which is straightforward. I think the second option is better because it invites further conversat

## 2.3 Question

**When deploying the LLM, which parameters have the greatest impact on the model output?**

**Answer:** The parameters that control how the next word (token) is chosen have the greatest impact on the output, along with the chat template and the context length:

- **`--temp` (temperature)** has the biggest effect on randomness. A low value (for example 0.1) makes the output predictable and repetitive, and a high value makes it more random and creative, but also more likely to drift off-topic. The default is 0.8, and DeepSeek recommends 0.6 for this model.
- **`--top-k` and `--top-p`** limit which candidate words the model can choose from. Top-k keeps only the k most probable words (default 40), and top-p keeps the smallest set of words whose combined probability reaches p. Smaller values make the answers more focused, and larger values allow more variety.
- **`--repeat-penalty`** controls repetition. The default of 1.0 means it is disabled, and a higher value discourages the model from repeating the same words or phrases.
- **`--chat-template`** decides how the conversation is formatted for the model. If the template does not match the model, the output is garbled or the model never stops answering.
- **`-c` (context length)** sets how much text the model can process at once. A longer context lets the model use more of the conversation, but it needs more memory, and a very short one cuts the conversation off.
- **`max_tokens` / `n_predict`** limits the length of the answer, so a long reasoning block can cut off the final answer.

In this lab, the raw `/completions` call at the default temperature drifted off-topic (it answered about a circle's area when asked to say "Hello"), while the chat endpoint, which applies the chat template, gave a sensible reply. This shows how much the sampling settings and the template affect the output.